<a href="https://colab.research.google.com/github/Santiago-Echeverri-Arteaga/Fisica_Computacional_2/blob/master/curso_2026_2/01_nivelacion_ml/10_regresion_regularizacion.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg"
       alt="Abrir en Colab"/>
</a>

# Regresión multilineal, Ridge y Lasso

**Pregunta guía:** ¿Cómo regularizamos una ley física aproximada?<br>
**Duración sugerida:** 4 horas.<br>
**Entorno:** CPU; datos incluidos o generados en memoria.

El orden de trabajo es siempre: problema → matemática → implementación
mínima → biblioteca → evaluación → interpretación física.


## Modelo y regularización

En mínimos cuadrados buscamos
$\min_{\boldsymbol\beta}\|\mathbf y-X\boldsymbol\beta\|_2^2$.
Ridge añade $\alpha\|\boldsymbol\beta\|_2^2$ y contrae coeficientes
correlacionados; Lasso añade $\alpha\|\boldsymbol\beta\|_1$ y puede
volver algunos exactamente cero. La penalización depende de la escala,
por lo que estandarizamos dentro del pipeline.

Simularemos el alcance de un proyectil con una corrección suave por
arrastre. No afirmamos que la corrección sea una solución exacta: es un
laboratorio de identificación de un modelo sustituto.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

SEMILLA = 42
rng = np.random.default_rng(SEMILLA)
n = 800
v0 = rng.uniform(10, 80, n)
theta = rng.uniform(np.deg2rad(10), np.deg2rad(80), n)
masa = rng.uniform(0.05, 2.0, n)
arrastre = rng.uniform(0.0, 0.06, n)
g = 9.81
alcance_vacío = v0**2 * np.sin(2 * theta) / g
factor = 1 / (1 + 5.0 * arrastre * v0 / masa)
alcance = alcance_vacío * factor + rng.normal(0, 2.0, n)

X = pd.DataFrame(
    {"v0": v0, "theta_rad": theta, "masa": masa, "arrastre": arrastre}
)
y = pd.Series(alcance, name="alcance_m")
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEMILLA
)
display(X.head())


## Modelos comparables

Los tres reciben las mismas variables polinomiales de grado 2, escala,
pliegues y test. `PolynomialFeatures` permite productos como
$v_0\,c/m$; sigue siendo una regresión lineal respecto a sus parámetros.


In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=SEMILLA)

def pipeline(modelo):
    return Pipeline(
        [
            ("polinomio", PolynomialFeatures(degree=2, include_bias=False)),
            ("escala", StandardScaler()),
            ("modelo", modelo),
        ]
    )

configuraciones = {
    "lineal": (pipeline(LinearRegression()), {}),
    "ridge": (pipeline(Ridge()), {"modelo__alpha": np.logspace(-4, 4, 17)}),
    "lasso": (
        pipeline(Lasso(max_iter=30_000)),
        {"modelo__alpha": np.logspace(-4, 1, 16)},
    ),
}

búsquedas = {}
filas = []
for nombre, (estimador, grilla) in configuraciones.items():
    búsqueda = GridSearchCV(
        estimador,
        grilla,
        scoring="neg_root_mean_squared_error",
        cv=cv,
        n_jobs=-1,
    ).fit(X_dev, y_dev)
    búsquedas[nombre] = búsqueda
    pred = búsqueda.predict(X_test)
    filas.append(
        {
            "modelo": nombre,
            "RMSE_CV": -búsqueda.best_score_,
            "RMSE_test": root_mean_squared_error(y_test, pred),
            "MAE_test": mean_absolute_error(y_test, pred),
            "R2_test": r2_score(y_test, pred),
            "hiperparámetros": búsqueda.best_params_,
        }
    )

comparación = pd.DataFrame(filas).set_index("modelo")
display(comparación)


In [ ]:
mejor_nombre = comparación["RMSE_CV"].idxmin()
mejor = búsquedas[mejor_nombre].best_estimator_
pred = mejor.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(y_test, pred, alpha=0.55)
límites = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
axes[0].plot(límites, límites, "k--")
axes[0].set(xlabel="alcance real", ylabel="predicción", title=mejor_nombre)
axes[1].scatter(pred, y_test - pred, alpha=0.55)
axes[1].axhline(0, color="k", linestyle="--")
axes[1].set(xlabel="predicción", ylabel="residuo", title="Diagnóstico de residuos")
plt.tight_layout()
plt.show()


## Lectura física y ejercicios

Un buen $R^2$ no valida la ley de arrastre: sólo indica que el sustituto
predice bien dentro de la distribución simulada. Inspeccione residuos
contra velocidad y arrastre para buscar estructura omitida.

- Básico: use sólo la fórmula de vacío como predictor y mida el error.
- Intermedio: grafique norma de coeficientes contra $lpha$.
- Reto: entrene con velocidades hasta 60 m/s y pruebe entre 60–80 m/s;
  discuta interpolación frente a extrapolación.
